# 합성곱 신경망 (CNN)

CNN은 "이미지 인식" 과 "음성 인식" 등 다양한 곳에 사용됩니다.

특히 이미지 인식 분야에서 딥러닝을 활용한 기법은 거의 다 CNN을 기초로 합니다.

============================================================================================

지금까지의 신경망 구조

- 인접하는 계층의 모든 뉴런과 결합되어 있습니다. (이를 완전연결 (Fully Connected)라고 합니다. )
- Affine Layer + ReLU (or Sigmoid) Layer로 구성되어 있습니다.

[입력 이미지] -> [Affine] -> [ReLU] (1 Layer) -> [Affine] -> [ReLU] (2 Layer)...-> [Affine] -> [SoftMax] -> 결과 출력 (최종 결과: 확률 )


============================================================================================

[CNN의 구조]

[입력 이미지] -> [Conv] -> [ReLU] -> [Pooling] -> [Conv] -> [ReLU] -> [Pooling] -> ... -> [Affine] -> [ReLU] -> [Affine] -> [Softmax] -> (최종 결과: 확률 )

- Conv Layer가 추가된 것을 확인할 수 있다.
- Pooling Layer도 추가되었지만 생략도 가능하다.


CNN의 필수 개념 소개

- 패딩 (padding) : 패딩은 주로 Conv 연산이나 Max pooling 시 발생하는 이미지의 축소를 방지하기 위해서 사용됩니다.
- 스트라이드 (stride): 커널의 이동 시, 얼만큼 이동할지를 지정하는 역할을 합니다.


Fully Connected Layer (FC Layer)의 기존 문제점:

Fully Connected Layer의 경우, 입력단에서 "데이터의 형상이 무시"됩니다.

예를 들어 입력단에 들어갈 수 있는 데이터의 형식은 (1, 28, 28) (channel, w_size, h_size)라고 하더라도 1차원 배열인 1* 28 * 28 = 784개의 데이터를 첫 번째 Layer에 입력합니다.

-> 이미지는 3차원 차원이면서 공간적 정보가 담겨져 있습니다.
- 이미지는 RGB 3차원
- Edge의 경우, 픽셀 값의 차이가 심함
- 배경이나 인접한 Pixel에서는 픽셀 값의 차이가 별로 없음

이러한 특징들을 FC Layer(Affine)에서는 무시하고 있기 때문에 FC Layer만으로는 이미지 데이터를 처리하기는 어렵다는 한계가 있습니다.

합성곱 신경망 (Convolution Layer) 사용

합성곱 신경망은 이미지도 3차원 데이터로 입력 받으며, 다음 계층에도 3차원 데이터로 전달합니다.

CNN에서는 합성곱 계층의 입출력 데이터를 특징 맵 (feature map)이라고 합니다.

합성곱 계층의 입력 데이터를 입력 특징 맵 (input feature map)이라고 하고 출력 데이터를 출력 특징맵 (output feature map)이라고 합니다.

합성곱 연산에서는 필터 연산을 수행합니다.

(4X4) 이미지에 (3X3) 커널을 적용하면 (2X2) 사이즈의 이미지로 출력됩니다.

합성곱 연산은 필터의 윈도우를 일정 간격으로 이동해가며 입력 데이터에 적용합니다.

입력과 필터에서 대응하는 원소끼리 곱한 후 그 총합을 구합니다. (이 계산을 단일 곱셈 -누산 (fused multiply -add, FMA)라고 합니다. )

완전연결 신경망에서는 가중치 매개변수와 편향이 존재했습니다.

CNN에서는 필터의 매개변수가 지금까지 배웠던 가중치에 해당합니다.

그리고 CNN에서도 편향이 존재합니다.

ex. (4x4)이미지에 커널(3x3) 사이즈를 적용 -> (2x2) 사이즈의 출력 결과 + 편향 -> 출력 데이터

==============================================================================

입력 이미지와 커널, 패딩, 스트라이드를 적용했을 때 나오는 출력 이미지의 가로x세로의 크기를 알아봅시다.

    공식은 다음과 같습니다.
    OH = (H + 2P -FH)/ S + 1
    OW = (W + 2P -FW)/ S +1

- OH = 출력의 높이
- OW = 출력의 너비
- H, W = 입력의 높이,너비
- P = 패딩
- FH, FW = 필터의 크기
- S = 스트라이드

합성곱 연산을 직육면체 블록으로 생각했을 때,

(채널, 높이, 너비) -> 3차원

하나의 필터 (C, FH, FW)를 적용하면 하나의 출력 (1, OH, OW)가 나오게 된다.

이유: 각각의 채널에 해당하는 (H,W)에 해당 채널의 필터 (FH,FW)를 적용하여 각 위치의 값을 더하는 방식으로 진행된다.

-> 그러면 합성곱 연산의 출력으로 다수의 채널을 내보내려면 어떻게 해야할까?

->답: 필터(가중치)를 다수 사용하면 됩니다.

입력 데이터 (C, H, W) * 필터 (FN, C, FH, FW) -> 출력 데이터 (FN, OH, OW)

FN = 필터의 수

다중 필터를 적용해서 해당 3차원 필터 하나당 하나의 출력 데이터가 나오므로

출력 데이터 (FN, OH, OW)에서 FN은 필터의 갯수만큼 나오게 됩니다.

배치 처리

합성곱 신경망에서는 입력 데이터를 한 덩어리로 묶어 배치로 처리합니다.

그래서 각 계층을 흐르는 데이터의 차원을 하나 늘려 4차원 데이터로 저장합니다. 구체적으로는 데이터를 (데이터 수, 채널 수, 높이, 너비) 순으로 저장합니다.

-> 데이터 N개에 대한 합성곱 연산이 이루어져 N회분의 처리를 한번에 수행합니다. (N개로 묶은 데이터를 한번에 처리한다는 이야기)

주의점:

입력 데이터의 N개의 데이터를 처리할 때 각각 하나의 입력 데이터에 여러 개의 필터를 제공하여 3차원의 출력 데이터로 만들고 이를 N번 반복하는 것과 같습니다.

풀링 계층:

커널 사이즈에 겹치는 값들중 가장 큰 값을 뽑는 max pooling

커널 사이즈에 겹치는 갑들의 평균을 추출하는 average pooling 등이 있습니다.

특징:

풀링을 진행하게 되면 공간 크기가 줄어듭니다.

이를 방지하고 싶을 때, padding을 사용합니다.

학습해야 할 매개변수가 없습니다.

    - 풀링 계층은 합성곱 계층과 달리 학습해야 할 매개변수가 없습니다.
    풀링은 대상 영역에서 최댓값이나 평균을 취하는 명확한 처리이므로 특별히 학습할 것이 없습니다.

채널 수가 변하지 않는다.

    - 풀링 연산은 입력 데이텅듸 채널 수 그대로 출력 데이터로 내보냅니다.

입력의 변화에 영향을 적게 받는다. (강건하다)

    - 입력 데이터가 조금 변해도 풀링의 결과는 잘 변하지 않습니다. (max pooling)의 경우 가장 높은 값이 엄청 변하지 않는 이상은 변함이 거의 없습니다.

In [2]:
import numpy as np

#합성곱/ 풀링 계층 구현하기

#4차원 배열
x = np.random.rand(10, 1, 28, 28) #무작위 데이터 생성 (10개, 1채널, H사이즈, W사이즈)
x.shape

(10, 1, 28, 28)

In [4]:
x[0].shape

(1, 28, 28)

In [5]:
x[1].shape

(1, 28, 28)

In [6]:
len(x)

10

In [9]:
#x[0,0] -> 첫 번째 데이터의 첫 채널의 공간 데이터
x[0,0].shape

(28, 28)

합성곱 연산을 수행하려면 for문을 겹겹이 써야합니다.

하지만 이렇게 할 경우, 넘파이에서 for문을 사용하면 성능이 떨어진다는 단점이 있습니다.

-> for문 대신 **im2col** 함수를 사용합니다.

im2col은 입력 데이터를 필터링 (가중치 계산) 하기 좋게 전개하는 함수입니다.

3차원의 입력 데이터를 2차원 행렬로 바꿉니다. (정확히는 배치까지 고려한 4차원 입력 데이터를 2차원 데이터로 변경합니다. )

ex.

4차원 입력 데이터에서 한 배치를 가져옵니다.

한 배치 = 3차원 데이터

1번째 채널의 (H,W)를 한 줄로 씁니다.

2번째 채널.. 3번째 채널까지도 똑같이 한 줄에 넣습니다.

이러면 메모장처럼 한 행에는 해당 채널의 값들이 한줄로 나열되는 것을 볼 수 있습니다.


**[과정 설명]**
1. 입력 데이터를 im2col을 통해 2차원 배열로 만듭니다.
2. 각각 4차원의 필터를 3차원의 필터 하나당 세로줄을 가진 2차원 데이터로 변경합니다.
3. 각각 입력 데이터(2차원) * 필터(2차원) 합성곱을 수행합니다. -> 그 결과 2차원의 출력 데이터

4. 해당 2차원 데이터를 다시 3차원으로 복원합니다.

In [6]:
# im2col 함수
def im2col(input_data, filter_h, filter_w, stride=1, pad=0):
    """다수의 이미지를 입력받아 2차원 배열로 변환한다(평탄화).

    Parameters
    ----------
    input_data : 4차원 배열 형태의 입력 데이터(이미지 수, 채널 수, 높이, 너비)
    filter_h : 필터의 높이
    filter_w : 필터의 너비
    stride : 스트라이드
    pad : 패딩

    Returns
    -------
    col : 2차원 배열
    """
    N, C, H, W = input_data.shape
    out_h = (H + 2*pad - filter_h)//stride + 1
    out_w = (W + 2*pad - filter_w)//stride + 1
    #패딩 적용하기
    img = np.pad(input_data, [(0,0), (0,0), (pad, pad), (pad, pad)], 'constant') #패딩 추가 부분 np.pad(데이터, [(0,0) : N에 패딩 안함, (0,0): Channel에 패딩 안함, (pad, pad): 위아래 pad만큼 0추가, (pad, pad) 좌우에 pad만큼 0 추가)])
    #저장할 사이즈에 0값 넣어두기
    col = np.zeros((N, C, filter_h, filter_w, out_h, out_w)) #출력 결과물을 저장할 공간을 만들어 두었습니다.

    for y in range(filter_h):
        y_max = y + stride*out_h    #해당 y값에서 처리해아할 영역 설정
        for x in range(filter_w):
            x_max = x + stride*out_w #해당 x값에서 처리해야할 영역 설정
            col[:, :, y, x, :, :] = img[:, :, y:y_max:stride, x:x_max:stride]   #col[0,1,2,3,4,5] 라는 값중 4,5번에 img[0,1,2,3]중 2,3번값을 각각 넣음

    col = col.transpose(0, 4, 5, 1, 2, 3).reshape(N*out_h*out_w, -1) #자리를 맞게 바꿔주고, reshape으로 행은 N*out_h*out_w 사이즈로 바꿈 (행사이즈를 해당 사이즈로 맞춤)
    return col

In [21]:
def col2im(col, input_shape, filter_h, filter_w, stride=1, pad=0):
    N, C, H, W = input_shape
    out_h = (H + 2*pad - filter_h) // stride + 1
    out_w = (W + 2*pad - filter_w) // stride + 1
    col = col.reshape(N, out_h, out_w, C, filter_h, filter_w).transpose(0, 3, 4, 5, 1, 2)

    img = np.zeros((N, C, H + 2*pad + stride - 1, W + 2*pad + stride - 1))
    for y in range(filter_h):
        y_max = y + stride * out_h
        for x in range(filter_w):
            x_max = x + stride * out_w
            img[:, :, y:y_max:stride, x:x_max:stride] += col[:, :, y, x, :, :]

    return img[:, :, pad:H + pad, pad:W + pad]

In [9]:
import numpy as np

#im2col 수행
#im2col(input_data, filter_h, filter_w, stride = 1, pad=0)

# import sys, os
# sys.path.append(os.path.join(os.path.dirname(__file__), '..'))

x1 = np.random.rand(1, 3, 7, 7) #1개, 3채널, h:7, w:7
col1 = im2col(x1, 5,5,stride=1, pad= 0)
print(col1.shape) #9,75

x2= np.random.rand(10, 3, 7,7) #데이터 10개
col2 = im2col(x2, 5,5,stride=1, pad=0)
print(col2.shape) #(90, 75)

(9, 75)
(90, 75)


In [22]:
#Convolution 클래스 구현
class Convolution:
    def __init__ (self, W, b, stride= 1, pad = 0):
        self.W = W
        self.b = b
        self.stride = stride
        self.pad = pad
        self.x = None
        self.col = None
        self.col_W = None
        self.dW = None
        self.db = None

    def forward(self, x):       #x는 입력 데이터
        FN, C, FH, FW = self.W.shape
        N, C, H, W = x.shape
        out_h = int (1 + (H + 2*self.pad - FH) // self.stride)
        out_w = int (1 + (W + 2*self.pad - FW) // self.stride)

        col = im2col(x, FH, FW, self.stride, self.pad)
        col_w = self.W.reshape(FN, -1).T        #변형: 원래는 FN마다 한줄을 가짐 -> .T로 바꿈 -> FN이라는 세로줄 갯수별로 하나에 -1 만큼을 가짐  -> 필터를 세로줄로 만듦
        out = np.dot (col, col_w) + self.b      #합성곱을 한 뒤에 self.b를 더해서 편향을 줌
        out = out.reshape(N, out_h, out_w, -1).transpose(0,3,1,2) #순서: (N, H,W, C) -> (N, C, H, W)로 바꿈

        self.x = x
        self.col = col
        self.col_W = col_w
        return out

    def backward(self, dout):
        FN, C, FH, FW = self.W.shape
        dout = dout.transpose(0, 2, 3, 1).reshape(-1, FN)

        self.db = np.sum(dout, axis=0)
        self.dW = np.dot(self.col.T, dout).transpose(1, 0).reshape(FN, C, FH, FW)
        dcol = np.dot(dout, self.col_W.T)
        return col2im(dcol, self.x.shape, FH, FW, self.stride, self.pad)

In [23]:
# pooling 계층 구현하기
class pooling:
    def __init__(self, pool_h, pool_w, stride= 1, pad = 0):
        self.pool_h = pool_h
        self.pool_w = pool_w
        self.stride = stride
        self.pad= pad
        self.x = None
        self.arg_max= None

    def forward(self, x):
        N, C, H, W = x.shape
        out_h = int(1+ (H-self.pool_h)/ self.stride)
        out_w = int(1+ (W-self.pool_w) / self.stride)

        #전개
        col = im2col(x, self.pool_h, self.pool_w, self.stride, self.pad)
        col = col.reshape(-1, self.pool_h*self.pool_w)
        self.arg_max = np.argmax(col, axis=1)
        #최댓값 구하기
        out = np.max(col, axis= 1)
        #성형
        out = out.reshape(N,out_h, out_w, C).transpose(0,3,1,2)
        self.x = x
        return out

    def backward(self, dout):
        dout = dout.transpose(0,2,3,1)
        pool_size= self.pool_h * self.pool_w
        dmax = np.zeros((dout.size, pool_size))

        # 최댓값 위치에만 gradient 전달
        dmax[np.arange(self.arg_max.size), self.arg_max.flatten()] = dout.flatten()
        dmax = dmax.reshape(dout.shape + (pool_size,))
        dcol = dmax.reshape(dmax.shape[0] * dmax.shape[1] * dmax.shape[2], -1)
        return col2im(dcol, self.x.shape, self.pool_h, self.pool_w, self.stride, self.pad)



In [24]:
class Relu:
    def __init__(self):
        self.mask = None

    def forward(self, x):
        self.mask = (x <=0)
        out = x.copy()
        out[self.mask] = 0
        return out

    def backward(self, dout):
        dout[self.mask] = 0
        return dout

In [17]:
class Affine:
    def __init__(self, W, b):
        self.W = W
        self.b = b
        self.x = None
        self.dW = None
        self.db = None

    def forward(self, x):
        self.x = x
        return np.dot(x, self.W) + self.b

    def backward(self, dout):
        self.dW = np.dot(self.x.T, dout)
        self.db = np.sum(dout, axis = 0)
        return np.dot(dout, self.W.T)

In [18]:
# SoftmaxWithLoss가 이 두 함수를 쓰는데 정의가 없어요!

def softmax(x):
    x = x - np.max(x, axis=-1, keepdims=True)  # 오버플로 방지
    return np.exp(x) / np.sum(np.exp(x), axis=-1, keepdims=True)

def cross_entropy_error(y, t):
    if y.ndim == 1:
        y = y.reshape(1, y.size)
        t = t.reshape(1, t.size)
    batch_size = y.shape[0]
    return -np.sum(t * np.log(y + 1e-7)) / batch_size

In [19]:
class SoftmaxWithLoss:
    def __init__(self):
        self.loss = None
        self.y = None
        self.t = None

    def forward(self, x, t):
        self.t = t
        self.y = softmax(x)
        self.loss = cross_entropy_error(self.y, self.t)
        return self.loss

    def backward(self, dout=1):
        batch_size = self.t.shape[0]
        return (self.y - self.t) / batch_size

In [25]:
from collections import OrderedDict
#CNN 구현하기

#초기화: 초기화 인수로 주어진 합성곱 계층의 하이퍼파라미터를 딕셔너리에서 꺼냅니다.
#그리고 합성곱 계층의 출력 크기를 계산합니다.
class SimpleconvNet:
    def __init__(self, input_dim = (1,28, 28),
                 conv_param = {'filter_num': 30, 'filter_size': 5,
                               'pad' : 0, 'stride': 1},
                 hidden_size= 100, output_size= 10, weight_init_std= 0.01):
        filter_num = conv_param['filter_num']
        filter_size= conv_param['filter_size']
        filter_pad = conv_param['pad']
        filter_stride= conv_param['stride']
        input_size= input_dim[1]
        conv_output_size = (input_size - filter_size + 2*filter_pad) / \
                            filter_stride +1
        pool_output_size = int(filter_num * (conv_output_size/2) *
                               (conv_output_size /2))

        #가중치 매개변수 초기화
        self.params = {}
        self.params['W1'] = weight_init_std * \
                            np.random.randn(filter_num, input_dim[0],
                                            filter_size, filter_size)
        self.params['b1'] = np.zeros(filter_num)
        self.params['W2'] = weight_init_std * \
                            np.random.randn(pool_output_size,
                                            hidden_size)
        self.params['b2'] = np.zeros(hidden_size)
        self.params['W3'] = weight_init_std * \
                            np.random.randn(hidden_size, output_size)
        self.params['b3'] = np.zeros(output_size)

        #CNN구성하는 계층들 생성
        self.layers= OrderedDict()
        self.layers['Conv1']= Convolution(self.params['W1'],
                                          self.params['b1'],
                                          conv_param['stride'],
                                          conv_param['pad'])
        self.layers['Relu1'] = Relu()
        self.layers['Pool1'] = pooling(pool_h= 2, pool_w = 2, stride= 2)
        self.layers['Affine1']= Affine(self.params['W2'],
                                       self.params['b2'])
        self.layers['Relu2'] = Relu()
        self.layers['Affine2'] = Affine(self.params['W3'],
                                        self.params['b3'])
        self.last_layer = SoftmaxWithLoss()

    def predict(self, x):
        for layer in self.layers.values():
            x = layer.forward(x)
        return x

    def loss(self, x, t):
        y= self.predict(x)
        return self.last_layer.forward(y, t)

In [26]:
def gradient(self, x, t):
    #순천파
    self.loss(x, t)

    #역전파
    dout= 1
    dout = self.last_layer.backward(dout)

    layers = list(self.layers.values())
    layers.reverse()
    for layer in layers:
        dout = layer.backward(dout)

    #결과 저장
    grads= {}
    grads['W1'] = self.layers['Conv1'].dW
    grads['b1'] = self.layers['Conv1'].db
    grads['W2'] = self.layers['Affine1'].dW
    grads['b2'] = self.layers['Affine1'].db
    grads['W3'] = self.layers['Affine2'].dW
    grads['b3'] = self.layers['Affine2'].db

    return grads

#전체 코드

In [20]:
import numpy as np
from collections import OrderedDict

def softmax(x):
    x = x - np.max(x, axis=-1, keepdims=True)
    return np.exp(x) / np.sum(np.exp(x), axis=-1, keepdims=True)

def cross_entropy_error(y, t):
    if y.ndim == 1:
        y = y.reshape(1, y.size)
        t = t.reshape(1, t.size)
    batch_size = y.shape[0]
    return -np.sum(t * np.log(y + 1e-7)) / batch_size

def im2col(input_data, filter_h, filter_w, stride=1, pad=0):
    N, C, H, W = input_data.shape
    out_h = (H + 2*pad - filter_h) // stride + 1
    out_w = (W + 2*pad - filter_w) // stride + 1
    img = np.pad(input_data, [(0,0), (0,0), (pad,pad), (pad,pad)], 'constant')
    col = np.zeros((N, C, filter_h, filter_w, out_h, out_w))
    for y in range(filter_h):
        y_max = y + stride * out_h
        for x in range(filter_w):
            x_max = x + stride * out_w
            col[:, :, y, x, :, :] = img[:, :, y:y_max:stride, x:x_max:stride]
    col = col.transpose(0, 4, 5, 1, 2, 3).reshape(N*out_h*out_w, -1)
    return col

class Relu:
    def __init__(self):
        self.mask = None

    def forward(self, x):
        self.mask = (x <= 0)
        out = x.copy()
        out[self.mask] = 0
        return out

    def backward(self, dout):
        dout[self.mask] = 0
        return dout

class Affine:
    def __init__(self, W, b):
        self.W = W
        self.b = b
        self.x = None
        self.dW = None
        self.db = None

    def forward(self, x):
        self.x = x
        return np.dot(x, self.W) + self.b

    def backward(self, dout):
        self.dW = np.dot(self.x.T, dout)
        self.db = np.sum(dout, axis=0)
        return np.dot(dout, self.W.T)

class Convolution:
    def __init__(self, W, b, stride=1, pad=0):
        self.W = W
        self.b = b
        self.stride = stride
        self.pad = pad

    def forward(self, x):
        FN, C, FH, FW = self.W.shape
        N, C, H, W = x.shape
        out_h = int(1 + (H + 2*self.pad - FH) // self.stride)
        out_w = int(1 + (W + 2*self.pad - FW) // self.stride)
        col = im2col(x, FH, FW, self.stride, self.pad)
        col_w = self.W.reshape(FN, -1).T
        out = np.dot(col, col_w) + self.b
        out = out.reshape(N, out_h, out_w, -1).transpose(0, 3, 1, 2)
        return out

class Pooling:
    def __init__(self, pool_h, pool_w, stride=1, pad=0):
        self.pool_h = pool_h
        self.pool_w = pool_w
        self.stride = stride
        self.pad = pad

    def forward(self, x):
        N, C, H, W = x.shape
        out_h = int(1 + (H - self.pool_h) / self.stride)
        out_w = int(1 + (W - self.pool_w) / self.stride)
        col = im2col(x, self.pool_h, self.pool_w, self.stride, self.pad)
        col = col.reshape(-1, self.pool_h * self.pool_w)
        out = np.max(col, axis=1)
        out = out.reshape(N, out_h, out_w, C).transpose(0, 3, 1, 2)
        return out

class SoftmaxWithLoss:
    def __init__(self):
        self.loss = None
        self.y = None
        self.t = None

    def forward(self, x, t):
        self.t = t
        self.y = softmax(x)
        self.loss = cross_entropy_error(self.y, self.t)
        return self.loss

    def backward(self, dout=1):
        batch_size = self.t.shape[0]
        return (self.y - self.t) / batch_size

class SimpleConvNet:
    def __init__(self, input_dim=(1, 28, 28),
                 conv_param={'filter_num': 30, 'filter_size': 5,
                             'pad': 0, 'stride': 1},
                 hidden_size=100, output_size=10, weight_init_std=0.01):
        filter_num    = conv_param['filter_num']
        filter_size   = conv_param['filter_size']
        filter_pad    = conv_param['pad']
        filter_stride = conv_param['stride']
        input_size    = input_dim[1]
        conv_output_size = (input_size - filter_size + 2*filter_pad) / filter_stride + 1
        pool_output_size = int(filter_num * (conv_output_size/2) * (conv_output_size/2))

        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(filter_num, input_dim[0], filter_size, filter_size)
        self.params['b1'] = np.zeros(filter_num)
        self.params['W2'] = weight_init_std * np.random.randn(pool_output_size, hidden_size)
        self.params['b2'] = np.zeros(hidden_size)
        self.params['W3'] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params['b3'] = np.zeros(output_size)

        self.layers = OrderedDict()
        self.layers['Conv1']   = Convolution(self.params['W1'], self.params['b1'],
                                             conv_param['stride'], conv_param['pad'])
        self.layers['Relu1']   = Relu()
        self.layers['Pool1']   = Pooling(pool_h=2, pool_w=2, stride=2)
        self.layers['Affine1'] = Affine(self.params['W2'], self.params['b2'])
        self.layers['Relu2']   = Relu()
        self.layers['Affine2'] = Affine(self.params['W3'], self.params['b3'])
        self.last_layer = SoftmaxWithLoss()

    def predict(self, x):
        for layer in self.layers.values():
            x = layer.forward(x)
        return x

    def loss(self, x, t):
        y = self.predict(x)
        return self.last_layer.forward(y, t)

    def gradient(self, x, t):
        self.loss(x, t)
        dout = 1
        dout = self.last_layer.backward(dout)
        layers = list(self.layers.values())
        layers.reverse()
        for layer in layers:
            dout = layer.backward(dout)
        grads = {}
        grads['W1'] = self.layers['Conv1'].dW
        grads['b1'] = self.layers['Conv1'].db
        grads['W2'] = self.layers['Affine1'].dW
        grads['b2'] = self.layers['Affine1'].db
        grads['W3'] = self.layers['Affine2'].dW
        grads['b3'] = self.layers['Affine2'].db
        return grads

In [ ]:
import numpy as np
from mnist import load_mnist  # MNIST 데이터 로드

# 1. 데이터 로드
(x_train, t_train), (x_test, t_test) = load_mnist(flatten=False)
# flatten=False → (N, 1, 28, 28) 형태로 불러옴

# 2. 모델 생성
network = SimpleConvNet(input_dim=(1, 28, 28),
                        conv_param={'filter_num': 30, 'filter_size': 5,
                                    'pad': 0, 'stride': 1},
                        hidden_size=100, output_size=10,
                        weight_init_std=0.01)

# 3. 학습 설정
learning_rate = 0.01
epochs = 10
batch_size = 100
train_size = x_train.shape[0]

# 4. 학습 루프
for epoch in range(epochs):
    # 미니배치 랜덤 선택
    batch_mask = np.random.choice(train_size, batch_size)
    x_batch = x_train[batch_mask]
    t_batch = t_train[batch_mask]

    # gradient 계산
    grads = network.gradient(x_batch, t_batch)

    # 가중치 업데이트
    for key in ('W1', 'b1', 'W2', 'b2', 'W3', 'b3'):
        network.params[key] -= learning_rate * grads[key]

    # loss 출력
    loss = network.loss(x_batch, t_batch)
    print(f'Epoch {epoch+1:2d}/{epochs} | Loss: {loss:.4f}')

# 5. 평가
correct = 0
for i in range(len(x_test)):
    y = network.predict(x_test[i:i+1])
    pred = np.argmax(y)
    if pred == t_test[i]:
        correct += 1

print(f'정확도: {100 * correct / len(x_test):.2f}%')

## PyTorch로 구현해보기! (조금 더 쉬운 버전)

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((32,32)),             #32x32크기로 통일
    transforms.ToTensor(),                  #텐서로 변환
    transforms.Normalize((0.5, 0.5, 0.5),   #0.0~1.0 사이의 픽셀값을 -1.0~1.0 사이로 Normalize -> 학습 안정화
                         (0.5, 0.5, 0.5))
])

trainset= torchvision.datasets.CIFAR10(
    root= '/data',
    train=True,
    download=True,
    transform=transform
)

trainloader= torch.utils.data.DataLoader(
    trainset,
    batch_size= 32,
    shuffle= True
)

testset= torchvision.datasets.CIFAR10(
    root='/data',
    train= False,
    download= True,
    transform = transform
)

testloader = torch.utils.data.DataLoader(
    testset,
    batch_size= 32,
    shuffle= False
)

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        #Conv층
        self.conv1= nn.Conv2d(3, 32, (3,3), (1,1),1)
        #Pool층
        self.pool = nn.MaxPool2d((2,2), 2)
        #FC 층
        self.fc1 = nn.Linear(8192, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        #Conv1 -> ReLU -> Pool -> Flatten -> fc1 -> ReLU -> fc2 -> 출력
        x= F.relu(self.conv1(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1) #Flatten
        x= F.relu(self.fc1(x))
        x= self.fc2(x)
        return x

#학습 설정
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer= torch.optim.Adam(params=model.parameters(), lr= 0.001)

#학습 루프
for epoch in range(10):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in trainloader:
        optimizer.zero_grad()
        outputs= model(images)
        loss= criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(trainloader)
    epoch_acc= 100 * correct / total
    print(f'Epoch {epoch+1:2d}/10 | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2d}%')


#평가
correct = 0
total = 0
model.eval()
with torch.no_grad():
    for images, labels in testloader:
        outputs= model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'정확도: {100 * correct / total:.2f}%')

100%|██████████| 170M/170M [00:13<00:00, 12.3MB/s]


Epoch 1 완료
Epoch 2 완료
Epoch 3 완료
Epoch 4 완료
Epoch 5 완료
Epoch 6 완료
Epoch 7 완료
Epoch 8 완료
Epoch 9 완료
Epoch 10 완료
정확도: 65.31%
